# CMP AI LAB SIM 학생용
**내 실험 CSV → 평균 제거율 예측 → 검사 방침 선택 → 모델 JSON → 관제탑**

이 노트북은 특성화고 학생용입니다. 위에서부터 ①~⑤의 실행 버튼을 누르세요. 설치나 GPU는 필요하지 않습니다.

- **SIM**: 시뮬레이터에서 수집한 자료. 먼저 60장 이상을 수집합니다.
- **REAL**: 제공된 `REAL_PHM2016_stage_A.csv`. 같은 방법을 별도 자료에 적용해 비교합니다.
- 두 출처를 섞지 않습니다. 평균 제거율을 예측하며, 최종 검사 정답은 입력으로 사용하지 않습니다.
- 한 행은 한 웨이퍼·공정 단계의 요약입니다. 센서 요약을 쓰므로 **공정 완료 후, 검사 확인 전** 예측입니다.

학교망이 막혔다면 교사 예시 모델로 관제탑 활동을 진행하고, 새 학습은 인터넷이 되는 환경에서 수행하세요.


In [ ]:
#@title ① 준비하기 · 코드 수정 없이 실행
"""CMP Colab / local training. No downloads, pickle loading, or external service needed."""
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import sklearn

FEATURES={'SIM':['pressure_kpa','rotation_rpm','pad_wear','slurry_ml_min'],
          'REAL':['pressure_scaled','rotation_scaled','pad_usage_scaled','slurry_scaled']}

def prepare(df):
    df=df.copy()
    required=['schema_version','source','wafer_id','run_id','stage','timestamp','mrr']
    missing=[c for c in required if c not in df]
    if missing:raise ValueError('필수 열이 없습니다: '+', '.join(missing))
    if not df['schema_version'].astype(str).eq('1.0.0').all():raise ValueError('자료 규격 버전은 1.0.0이어야 합니다.')
    sources=df['source'].dropna().unique().tolist()
    if len(sources)!=1 or sources[0] not in FEATURES:raise ValueError('SIM 또는 REAL 한 출처만 업로드하세요. 두 출처를 섞지 않습니다.')
    source=sources[0];features=FEATURES[source]
    stages=df['stage'].dropna().unique().tolist()
    if len(stages)!=1 or stages[0]!=('EDU' if source=='SIM' else 'A'):raise ValueError('SIM은 EDU, REAL은 A 단계 자료만 사용하세요.')
    if df[required].isna().any().any():raise ValueError('식별자·시간·정답에 빈칸이 있습니다.')
    if df['run_id'].astype(str).duplicated().any():raise ValueError('중복 run_id가 있습니다. 같은 자료를 다시 붙였는지 확인하세요.')
    for c in features+['mrr']:
        if c not in df:raise ValueError('필수 열이 없습니다: '+c)
        df[c]=pd.to_numeric(df[c],errors='coerce')
        if not np.isfinite(df[c]).all():raise ValueError(c+'의 빈칸·문자·무한값을 확인하세요. 자동으로 0을 넣지 않습니다.')
    if (df['mrr']<=0).any():raise ValueError('평균 제거율 정답은 양수여야 합니다. 출처와 계산을 확인하세요.')
    if source=='SIM':
        for c,low,high in [('duration_s',30,75),('initial_thickness_nm',1000,1000)]:
            if c not in df:raise ValueError('필수 열이 없습니다: '+c)
            values=pd.to_numeric(df[c],errors='coerce')
            if not values.between(low,high).all():raise ValueError(c+'의 교육용 범위를 확인하세요.')
        if not df.timestamp.astype(str).str.contains(r'(?:Z|[+-]\d{2}:\d{2})$',regex=True).all():raise ValueError('SIM 시각에 시간대가 필요합니다.')
    # REAL timestamp is a proprietary scaled ordering value, not a calendar date.
    if source=='REAL':df['_order']=pd.to_numeric(df['timestamp'],errors='coerce')
    else:df['_order']=pd.to_datetime(df['timestamp'],errors='coerce',utc=True)
    if df['_order'].isna().any():raise ValueError('시간/순서 값을 확인하세요.')
    groups=df.groupby('wafer_id',sort=False)['_order'].agg(['min','max']).sort_values(['min','max'],kind='stable')
    if len(groups)<30:raise ValueError('학습·검증·시험을 나누려면 서로 다른 웨이퍼가 30개 이상 필요합니다. 시뮬레이터의 실험표 60장을 수집하세요.')
    n=len(groups);a=n*70//100;b=n*85//100
    mapping={key:('train' if i<a else 'validation' if i<b else 'test') for i,key in enumerate(groups.index)}
    df['split']=df['wafer_id'].map(mapping)
    # A group that crosses a chronological boundary must not leak later records into training.
    for first,second in [('train','validation'),('validation','test')]:
        if df.loc[df.split==first,'_order'].max()>df.loc[df.split==second,'_order'].min():
            raise ValueError('동일 웨이퍼의 시간 구간이 분할 경계를 가로지릅니다. 웨이퍼별 한 행으로 요약하거나 경계를 다시 정하세요.')
    return df,source,features,stages[0]

def predict_json(model,frame):
    X=frame[model['features']].to_numpy(dtype=np.float32)
    output=[]
    for row in X:
        total=0.0
        for t in model['trees']:
            i=0;steps=0
            while t['left'][i]!=-1:
                steps+=1
                if steps>50:raise ValueError('이전 모델의 나무 구조가 잘못되었습니다.')
                i=t['left'][i] if row[t['feature'][i]]<=t['threshold'][i] else t['right'][i]
            total+=t['value'][i]
        output.append(total/len(model['trees']))
    return np.array(output)

def fit_and_export(frame,outdir,version='student-v1'):
    import re
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,60}',str(version)):raise ValueError('모델 버전은 영문·숫자·밑줄·하이픈 1~60자로 적으세요.')
    df,source,features,stage=prepare(frame)
    train=df[df.split=='train'];val=df[df.split=='validation'];test=df[df.split=='test']
    rf=RandomForestRegressor(n_estimators=48,max_depth=8,min_samples_leaf=2,random_state=42,n_jobs=1)
    rf.fit(train[features],train.mrr)
    baseline=float(train.mrr.mean());metrics={}
    for name,data in [('validation',val),('test',test)]:
        p=rf.predict(data[features]);metrics[name]={'n':len(data),'mae':float(mean_absolute_error(data.mrr,p)),'baseline_mae':float(mean_absolute_error(data.mrr,np.full(len(data),baseline)))}
    model={'format':'daa-cmp-rf-v1','schema_version':'1.0.0','source':source,'stage':stage,'features':features,'target':'mrr','target_unit':'nm/min (educational)' if source=='SIM' else 'proprietary_scaled','inference_timing':'post_process_pre_inspection','model_version':str(version)[:80],'sklearn_version':sklearn.__version__,'training_ranges':{f:[float(train[f].min()),float(train[f].max())] for f in features},'split_counts':df.split.value_counts().to_dict(),'metrics':metrics,'split_rule':'wafer groups by time 70/15/15; training only; validation for policy; test once','trees':[]}
    from datetime import datetime,timezone
    model.update(model_id=source+'-'+version,source_type=source,feature_columns=features,feature_types={f:'number' for f in features},imputer_params={'strategy':'reject_missing','scaling':'none'},target_name='mrr',training_dataset_hash=hashlib.sha256(train.drop(columns=['_order']).to_csv(index=False).encode()).hexdigest(),trained_at=datetime.now(timezone.utc).isoformat(),model_format='daa-cmp-rf-v1',model_parameters={'n_estimators':48,'max_depth':8,'min_samples_leaf':2,'random_state':42},input_cast='float32')
    for est in rf.estimators_:
        t=est.tree_;model['trees'].append({'left':t.children_left.tolist(),'right':t.children_right.tolist(),'feature':t.feature.tolist(),'threshold':t.threshold.tolist(),'value':t.value[:,0,0].tolist()})
    all_pred=rf.predict(df[features]);delta=float(np.max(np.abs(all_pred-predict_json(model,df))))
    if delta>1e-9:raise RuntimeError('모델 저장 후 예측 검증에 실패했습니다.')
    model['export_max_abs_error']=delta
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    path=out/f'{source}_model_{version}.json';path.write_text(json.dumps(model,ensure_ascii=False,allow_nan=False,separators=(',',':')),encoding='utf-8')
    df['predicted_mrr']=all_pred;df['absolute_error']=abs(df.mrr-all_pred)
    if source=='SIM':
        df['predicted_thickness_nm']=pd.to_numeric(df.initial_thickness_nm)-all_pred*pd.to_numeric(df.duration_s)/60
    clean=df.drop(columns=['_order'])
    clean.to_csv(out/f'{source}_predictions_{version}.csv',index=False,encoding='utf-8-sig')
    clean[clean.split=='validation'].to_csv(out/f'{source}_validation_{version}.csv',index=False,encoding='utf-8-sig')
    clean[clean.split=='test'].to_csv(out/f'{source}_test_{version}.csv',index=False,encoding='utf-8-sig')
    (out/f'{source}_metrics_{version}.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2))
    return model,clean,metrics


import os, io, zipfile
try:
    from google.colab import files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
print('준비 완료. 다음 셀에서 CSV 한 개를 선택하세요.')
print('scikit-learn 버전:', sklearn.__version__)


## ② 데이터 선택
실험실에서 내려받은 CSV 하나를 선택합니다. 예비 실습은 `SIM_training.csv`, 공개 실측 비교는 `REAL_PHM2016_stage_A.csv`를 사용합니다. 판정 기록 CSV는 학습 원자료와 열 구성이 다릅니다.


In [ ]:
#@title ② CSV 한 개 불러오기
if IN_COLAB:
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('CSV 한 개만 선택하세요.')
    filename=next(iter(uploaded))
    if not filename.lower().endswith('.csv'):raise ValueError('CSV 파일이 필요합니다.')
    frame=pd.read_csv(io.BytesIO(uploaded[filename]))
else:
    filename=os.environ.get('CMP_DATA_PATH','SIM_my_experiments.csv')
    frame=pd.read_csv(filename)
print('파일:',Path(filename).name,' / 행:',len(frame))


## ③ 입력과 정답 구분
압력·회전·패드 사용·슬러리는 **입력**, `mrr`은 검사로 구한 **정답**입니다. 웨이퍼 ID, 순서, 검사 두께는 AI 입력에 넣지 않습니다.

웨이퍼를 시간순으로 정렬하여 학습 70%, 검증 15%, 시험 15%로 나눕니다. 장수는 정수로 나누므로 비율은 조금 달라질 수 있습니다. 같은 웨이퍼를 여러 묶음에 나누지 않습니다. 전처리를 정할 때에도 학습 자료만 사용합니다. 빈칸은 자동으로 0으로 채우지 않습니다.


In [ ]:
#@title ③ 자료와 분할 확인
prepared,source,features,stage=prepare(frame)
if source!='SIM':raise ValueError('이 노트북은 SIM 자료용입니다.')
print('출처:',source,' / 단계:',stage)
print('입력 열:',', '.join(features))
print('정답 열: mrr')
print(prepared.groupby('split',sort=False).size().rename('웨이퍼 수').to_string())
print('\n자료 첫 3행')
display(prepared[['run_id']+features+['mrr','split']].head(3))
print('SIM 단위: 수업용 nm/min' if source=='SIM' else 'REAL 단위: 원공개자가 변환한 척도. nm/min으로 해석하지 않습니다.')


## ④ 기본 모델 하나 학습
여러 작은 결정나무의 예측을 평균하는 **랜덤 포레스트 회귀**를 사용합니다. 학생은 나무 수를 바꾸는 경쟁보다 자료와 결과를 해석합니다.

MAE는 예측과 정답 사이 거리의 평균입니다. 항상 학습 자료의 평균만 말하는 기준 모델과 비교하세요. AI가 기준 모델보다 나쁘면 그 결과도 기록합니다. 시험 자료를 보고 모델 설정을 다시 맞추지 않습니다.


In [ ]:
#@title ④ 학습하고 검증 오차 보기
MODEL_VERSION = "student-v1" #@param {type:"string"}
output_dir=Path('cmp_output')/MODEL_VERSION
model,predictions,metrics=fit_and_export(frame,output_dir,MODEL_VERSION)
v=metrics['validation']
print('검증 자료:',v['n'],'장')
print('AI 검증 MAE:',round(v['mae'],3))
print('평균 기준 모델 MAE:',round(v['baseline_mae'],3))
print('이 자료에서는 AI의 평균 오차가 더 작습니다.' if v['mae']<v['baseline_mae'] else '이 자료에서는 AI가 평균 기준 모델보다 낫지 않습니다. 자료의 변화·표본·입력 정보의 한계를 검토하세요.')
print('시험 결과는 검사 방침을 정한 뒤 다음 셀에서 확인합니다.')
display(predictions[predictions.split=='validation'][['run_id','mrr','predicted_mrr','absolute_error']].head(5))

def policy_table(data):
    results=[]
    for name,low,high in [('A',485,515),('B',490,510)]:
        counts={'방침':name,'통상 검사':0,'추가 검사':0,'보류':0,'통상 검사 중 실제 부적합':0}
        for _,row in data.iterrows():
            thick=float(row.predicted_thickness_nm)
            outside=any(float(row[f])<model['training_ranges'][f][0] or float(row[f])>model['training_ranges'][f][1] for f in features)
            status='보류' if thick<480 or thick>520 else ('추가 검사' if outside or thick<low or thick>high else '통상 검사')
            counts[status]+=1
            if status=='통상 검사' and not 480<=float(row.inspection_thickness_nm)<=520:counts['통상 검사 중 실제 부적합']+=1
        results.append(counts)
    return pd.DataFrame(results)

if source=='SIM':
    print('검증 자료 A/B 비교: 다음 셀 실행 전에 방침을 선택하세요.')
    display(policy_table(predictions[predictions.split=='validation']))


## ⑤ 검사 방침을 정하고 마지막 시험 확인
SIM 품질 규격은 **480~520 nm로 고정**합니다. A의 통상 검사 구간은 485~515, B는 490~510 nm입니다. 규격 안이지만 통상 구간 밖이면 추가 검사, 규격 밖이면 보류입니다. 학습 범위 밖 입력은 최소한 추가 확인 대상으로 봅니다. 통상 검사도 검사를 생략하지 않습니다.

먼저 ④ 실행 결과의 **검증 A/B 표**를 보고 활동지에 선택 이유를 씁니다. 방침을 바꿀 때는 검증 표만 근거로 사용합니다. 시험 자료는 최종 확인으로 해석하며, 결과가 마음에 들지 않는다고 재분할하지 않습니다. REAL에는 실제 검사 합격 규격이 없으므로 제거율 오차만 확인합니다.


In [ ]:
#@title ⑤ 검증 방침 비교와 최종 시험 · 결과 ZIP 저장
POLICY = "A" #@param ["A", "B"]
REASON = "활동지에 검증 표를 근거로 선택 이유를 기록합니다." #@param {type:"string"}
if source=='SIM':
    print('검증 자료 A/B 비교')
    print('④의 검증 표로 방침을 선택한 뒤 시험 결과를 확인합니다.')
    print('선택 방침:',POLICY,' / 근거:',REASON)
else:
    print('REAL은 품질 방침 대신 공개 제거율 척도의 오차를 비교합니다.')
t=metrics['test']
print('\n마지막 시험:',t['n'],'장 / AI MAE:',round(t['mae'],3),' / 평균 기준 MAE:',round(t['baseline_mae'],3))
print('이 시험 결과는 해당 자료의 결과이며, 실제 공장 적용 성능이나 학생 학습 효과를 뜻하지 않습니다.')
(output_dir/'my_decision.json').write_text(json.dumps({'source':source,'policy':POLICY if source=='SIM' else None,'reason':REASON,'model_version':MODEL_VERSION},ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path(f'CMP_{source}_{MODEL_VERSION}_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(output_dir.iterdir()):z.write(f,f.name)
print('관제탑에 불러올 모델:',source+'_model_'+MODEL_VERSION+'.json')
print('학습 자료와 다른 새 실험 CSV를 관제탑에 접수하세요.')
if IN_COLAB:files.download(str(archive))
else:print('저장:',archive.resolve())


## 출처와 구현 범위
- 공정·원자료: [PHM Society 2016 공식 안내](https://phmsociety.org/conference/annual-conference-of-the-phm-society/annual-conference-of-the-prognostics-and-health-management-society-2016/phm-data-challenge-4/)
- [공식 데이터 설명서](https://phmsociety.org/wp-content/uploads/2016/05/PHM16DataChallengeCFP.pdf)
- 데이터 확보 경로: [공개 재배포 저장소](https://github.com/akangel0307/PHM-Data-Challenge). 수업용 CSV의 추출 이력과 원파일 해시는 자료 묶음의 `REAL_source_manifest.json`에 있습니다.
- 모델: [scikit-learn RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

REAL 자료는 특정 국내 반도체 회사의 자료로 확인되지 않았습니다. 모든 공개 변환값은 원척도로 유지합니다. SIM은 압력·회전·패드와 제거량 관계를 학습하는 단순 모의 실험이며, 실제 공장 수율이나 전체 표면 평탄도를 검증하지 않습니다.
